In [1]:
import requests
import os
import concurrent.futures
from rich.progress import (
    Progress,
    BarColumn,
    DownloadColumn,
    TransferSpeedColumn,
    TextColumn,
    TimeRemainingColumn,
)

def parse_links_from_file(filename):
    try:
        with open(filename, 'r') as file:
            return [line.strip() for line in file if line.strip()]
    except FileNotFoundError:
        return []

def download_file_worker(task_id, progress, url, save_path):
    """
    Worker function that updates a specific progress bar (task_id).
    """
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        total_length = int(response.headers.get('content-length', 0))
        
        # Update the progress bar with the correct total size and start it
        progress.update(task_id, total=total_length)
        progress.start_task(task_id)

        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                # Advance the specific bar by the chunk size
                progress.update(task_id, advance=len(chunk))
                
    except Exception as e:
        print(f"Error downloading {url}: {e}")

# --- Main Execution ---
if __name__ == "__main__":
    links_file = r"D:\Datasets\Imagenes Satelitales\Estados Unidos\NY State Imagery\Rest of NY State links (only high density).txt"
    output_folder = r"D:\Datasets\Imagenes Satelitales\Estados Unidos\NY State Imagery\Rest of NY"
    os.makedirs(output_folder, exist_ok=True)
    
    urls = parse_links_from_file(links_file)

    # Configure the Visuals
    progress = Progress(
        TextColumn("[bold blue]{task.fields[filename]}", justify="right"),
        BarColumn(bar_width=None),
        "[progress.percentage]{task.percentage:>3.1f}%",
        "•",
        DownloadColumn(),
        "•",
        TransferSpeedColumn(),
        "•",
        TimeRemainingColumn(),
    )

    with progress:
        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
            futures = []
            for url in urls:
                filename = url.split("/")[-1]
                save_path = os.path.join(output_folder, filename)
                if os.path.exists(save_path):
                    continue

                # Add a new bar for this file (start=False so it waits for connection)
                task_id = progress.add_task("download", filename=filename, start=False)
                
                # Submit the job
                futures.append(executor.submit(download_file_worker, task_id, progress, url, save_path))
            
            # Wait for all to finish
            concurrent.futures.wait(futures)

Output()